# Laboratorio 3: Sistema de Búsqueda
## Estudio experimental del escalamiento de listas, árboles binarios de búsqueda y árboles B+

- **Curso:** Estructuras de datos y laboratorio.
- **Autor:** Daniel Sánchez Escobar 
- **Fecha:** [fecha de entrega]  

### Resumen
Este informe estudia experimentalmente cómo cambia el tiempo de ejecución de un sistema de
administración de estudiantes según la estructura de datos que lo soporta: una lista de Python,
un árbol binario de búsqueda (ABB) y un árbol B+. Se miden las operaciones de búsqueda por ID,
inserción, listado ordenado y búsqueda por rango, variando el tamaño de los datos (N), el orden
de inserción (aleatorio u ordenado) y otros parámetros. Los resultados se comparan con la
complejidad teórica de cada estructura.

**Hallazgos principales:** [se completa al final, con los resultados obtenidos].

> **Uso de IA:** este trabajo se desarrolló con asistencia de herramientas de IA, autorizada por el
> enunciado del laboratorio. El detalle está en la sección final "Uso de IA y código de honor".

## 1. Problema y objetivo

### El problema
Una institución educativa necesita administrar los registros de sus estudiantes. Cada registro
contiene un **ID** único de matrícula, un **nombre**, una **edad** y un **promedio** académico.
El sistema debe permitir:

1. **Buscar** un estudiante a partir de su ID.
2. **Insertar** un nuevo estudiante.
3. **Listar** todos los estudiantes en orden ascendente de ID.

Además de las tres operaciones del enunciado, se implementó la **búsqueda por rango de ID**
(todos los estudiantes con ID entre `lb` y `ub`), que permite comparar las estructuras en una
operación donde el orden de los datos es especialmente relevante.

Los registros usados en los experimentos son **sintéticos**: los IDs, edades y promedios se
generan con semillas fijas (reproducibles) y los nombres se toman de un conjunto generado con
la librería Faker.

### Objetivo
Estudiar experimentalmente cómo las distintas estrategias de almacenamiento y búsqueda
(lista, ABB y árbol B+) afectan el tiempo de ejecución a medida que crece el tamaño de los
datos, y contrastar los resultados con la complejidad teórica. Las conclusiones se sustentan
en mediciones repetidas y análisis estadístico.

### Preguntas que guían el estudio
- ¿Los tiempos medidos se comportan como predice la complejidad teórica?
- ¿Qué ocurre con el ABB cuando los datos se insertan ordenadamente, y qué relación hay entre la altura del árbol y el tiempo de búsqueda?
- ¿Qué diferencias aparecen entre datos aleatorios y ordenados?
- ¿A partir de qué tamaño de entrada se vuelven claramente visibles las diferencias entre estructuras?
- ¿Existen costos constantes que hagan que algoritmos de distinta complejidad tengan tiempos similares con entradas pequeñas?

### Variables del estudio
| Variable | Valores |
|---|---|
| Estructura | Lista, ABB, árbol B+ |
| Tamaño de los datos (N) | [valores finales] |
| Número de búsquedas por medición (M) | [valor] |
| Orden de inserción | IDs aleatorios, IDs ordenados |
| Tamaño del resultado en rangos (k) | [valor(es)] |
| Orden del árbol B+ | [valor(es)] |

## 2. Estructuras de datos y algoritmos

Las tres estructuras implementan la misma interfaz (clase abstracta `Estructura`) con las
operaciones `insertar`, `buscar`, `listar` y `buscar_rango_id`, de modo que las mediciones
comparan implementaciones intercambiables. Las clases están en archivos `.py` del repositorio
y se importan en este notebook.

**Notación:** 
- N = número de estudiantes almacenados 
- h = altura del árbol
- k = número de estudiantes que devuelve una consulta por rango
- n = orden del árbol B+.

### 2.1 Lista de Python (`GestorListaEstudiantes`)
Los estudiantes se guardan en una lista nativa, en el orden en que llegan.
- **Insertar:** `append` al final, sin comparar con los elementos existentes.
- **Buscar:** recorrido lineal hasta encontrar el ID (o terminar la lista si no existe).
- **Listar:** como la lista no está ordenada, se ordena una copia con `sorted` (Timsort) y luego
  se recorre. Timsort aprovecha los tramos ya ordenados, así que con datos ya ordenados cuesta
  cerca de O(N).
- **Rango:** se recorre toda la lista filtrando los IDs dentro de [lb, ub] y solo se ordenan las
  coincidencias.

### 2.2 Árbol binario de búsqueda (`ABB`)
Cada nodo guarda un estudiante y referencias a sus hijos izquierdo y derecho. Los IDs menores o
iguales van a la izquierda y los mayores a la derecha. El árbol **no se balancea**, su forma
depende del orden de inserción. Con IDs aleatorios la altura esperada es de orden O(log N), mientras que con IDs
insertados en orden creciente el árbol se degenera en una cadena de altura N.

Los algoritmos de **inserción** y **búsqueda** son las versiones iterativas presentadas en
*Introduction to Algorithms* (Cormen, Leiserson, Rivest y Stein), traducidas a Python. El
**recorrido in-orden** para listar parte del algoritmo del mismo libro, que es recursivo, pero aquí se
implementó de forma **iterativa con pila explícita**.

- **Insertar y buscar:** descenso iterativo desde la raíz, con costo proporcional a la altura h.
- **Listar:** recorrido in-orden iterativo con pila explícita (evita el límite de recursión de
  Python en árboles muy profundos), que visita los IDs en orden ascendente.
- **Rango:** descenso a la primera clave ≥ lb y recorrido in-orden, descartando subárboles
  fuera del rango y deteniéndose al superar ub.
- **Altura:** recorrido por niveles, usado en el experimento que relaciona altura y tiempo.

### 2.3 Árbol B+ (`ArbolBPlus`)
Árbol balanceado de orden n: cada nodo tiene como máximo n punteros y n−1 claves. Los datos
están solo en las **hojas**, que además se enlazan entre sí en una lista ligada. Los nodos
internos solo guardan claves de búsqueda. Todas las hojas están a la misma profundidad.

Los algoritmos de **búsqueda**, **inserción** (con división de hojas y nodos internos) y
**búsqueda por rango** son los presentados en *Database System Concepts* (Silberschatz, Korth y Sudarshan), traducidos a Python.

- **Insertar:** se desciende a la hoja correspondiente. Si está llena, se divide en dos y la
  clave separadora sube al padre (que puede dividirse a su vez, hasta llegar a crear una nueva raíz).
- **Buscar:** descenso desde la raíz hasta la hoja y búsqueda de la clave dentro de ella. En cada
  nodo la búsqueda es lineal, así que el costo también depende del orden n.
- **Listar:** se baja a la hoja más a la izquierda y se recorren las hojas enlazadas, sin volver
  a subir por el árbol.
- **Rango:** descenso hasta la hoja de `lb` y recorrido de las hojas enlazadas hasta superar `ub`.

### 2.4 Complejidad teórica (la predicción que se contrastará con los datos)
| Estructura | Insertar | Buscar | Listar | Rango |
|---|---|---|---|---|
| Lista | O(1) | O(N) para búsqueda lineal | O(N log N) debido al ordenamiento previo | O(N + k log k) |
| ABB, IDs aleatorios | O(log N) esperado (equilibrado) | O(log N) esperado (equilibrado) | O(N) (recorrer el total de nodos)| O(log N + k) esperado |
| ABB, IDs ordenados | O(N) por inserción (O(N²) construir todo) | O(N) | O(N) | O(N + k) |
| Árbol B+ (orden fijo) | O(log N) | O(log N) | O(N) | O(log N + k) |

A continuación, en cada experimento se comprueba si los tiempos
medidos la respaldan o no.

## 3. Entorno de ejecución

Todos los tiempos de este informe se midieron en el equipo y con el software descritos a
continuación. Los resultados son válidos para este entorno: en otro equipo cambiarán los
valores absolutos, aunque no necesariamente las tendencias. Las versiones exactas de las
librerías están en `requirements.txt`.

In [ ]:
import sys
import platform
import subprocess
from importlib import metadata

import psutil


def nombre_cpu() -> str:
    """
    Devuelve el nombre comercial del procesador. platform.processor() suele venir
    vacío o genérico, así que se consulta según el sistema operativo: registro de
    Windows, sysctl en macOS o /proc/cpuinfo en Linux.

    Origen: generada con asistencia de IA (Claude, Anthropic, octubre 2026),
    autorizado por el enunciado del Laboratorio 3.
    Verificación: se validó que efectivamente coincide con las características del sistema.
    """
    try:
        if platform.system() == "Windows":
            import winreg
            clave = winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE,
                                   r"HARDWARE\DESCRIPTION\System\CentralProcessor\0")
            return winreg.QueryValueEx(clave, "ProcessorNameString")[0].strip()
        if platform.system() == "Darwin":
            return subprocess.check_output(
                ["sysctl", "-n", "machdep.cpu.brand_string"], text=True).strip()
        with open("/proc/cpuinfo") as archivo:
            for linea in archivo:
                if linea.startswith("model name"):
                    return linea.split(":", 1)[1].strip()
    except Exception:
        pass
    return platform.processor() or "No disponible"


# Hardware
memoria = psutil.virtual_memory()
print(f"CPU: {nombre_cpu()}")
print(f"Núcleos: {psutil.cpu_count(logical=False)} físicos, {psutil.cpu_count(logical=True)} lógicos")
print(f"RAM: {memoria.total / 1024**3:.1f} GB (disponible al iniciar: {memoria.available / 1024**3:.1f} GB)")

# Software
print(f"Sistema operativo: {platform.system()} {platform.release()} ({platform.machine()})")
print(f"Python: {platform.python_version()}")
for libreria in ["faker", "psutil", "numpy", "pandas", "matplotlib"]:   # ajusta a las que uses
    try:
        print(f"  {libreria}: {metadata.version(libreria)}")
    except metadata.PackageNotFoundError:
        print(f"  {libreria}: no instalada")

# Condiciones de la medición 
print("Condiciones: portátil, [conectado a corriente: sí], modo de energía [Mayor rendimiento], "
      "programas abiertos: [VS Code]")

# Versiones exactas de todo lo instalado, para reconstruir el entorno
with open("requirements.txt", "w", encoding="utf-8") as archivo:
    archivo.write(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))

CPU: AMD Ryzen 5 5500U with Radeon Graphics
Núcleos: 6 físicos, 12 lógicos
RAM: 7.3 GB (disponible al iniciar: 1.1 GB)
Sistema operativo: Windows 11 (AMD64)
Python: 3.14.7
  faker: 40.1.2
  psutil: 7.2.2
  numpy: 2.5.2
  pandas: no instalada
  matplotlib: no instalada
Condiciones: portátil, [conectado a corriente: sí], modo de energía [Mayor rendimiento], programas abiertos: [VS Code]


## 4. Parámetros globales

Los valores de esta sección fueron **sugeridos con asistencia de IA** (Claude, Anthropic).
[Si los ajustas después de probar, escribe qué cambiaste y por qué.]
Se definen en una sola celda para que cualquiera pueda modificarlos y repetir los experimentos.
Cómo se generan los datos, cómo se mide y qué estadísticas se calculan se explica en la
sección 5 (Metodología).

| Parámetro | Valor | Por qué |
|---|---|---|
| Tamaños de datos (N) | 100, 1.000, 10.000, 100.000, 1.000.000 | Cada tamaño es 10 veces el anterior. Esto cubre cuatro órdenes de magnitud con pocos puntos. Incluye 10.000, el caso del enunciado. |
| Búsquedas por medición (M) | 1.000 | Una sola búsqueda es demasiado rápida para medirla con fiabilidad, mil hacen que el tiempo total sea medible incluso con N pequeño. |
| Búsquedas exitosas | 80 % | La mayoría busca IDs que existen, y el 20 % restante busca IDs inexistentes, para incluir también el costo de una búsqueda fallida. |
| Repeticiones | 30 | Suficientes para calcular promedio y desviación estándar. |
| Calentamiento | 3 ejecuciones descartadas | Las primeras ejecuciones suelen ser más lentas y no son representativas. |
| Semilla base | 1000 | La repetición *i* usa la semilla de aleatorización `1000 + i`, de modo que cada medición utiliza datos distintos. |
| Orden del árbol B+ | 4 | Es el orden que se usó al implementarlo. |
| Tamaño del rango (k) | 50 | Cada búsqueda por rango devuelve exactamente 50 estudiantes, sin importar el tipo de datos. |



In [ ]:
# Parámetros globales (valores sugeridos con asistencia de IA: Claude, Anthropic, octubre 2026)
TAMANOS_N = [100, 1_000, 10_000, 100_000, 1_000_000]
M_BUSQUEDAS = 1_000
PROPORCION_EXITOSAS = 0.8
REPETICIONES = 30
CALENTAMIENTO = 3
SEMILLA_BASE = 1000
ORDEN_BPLUS = 4
K_RANGO = 50

## 5. Metodología

> Sección redactada con asistencia de IA (Claude, Anthropic) y revisada por mí.

El estudio mide el **tiempo de ejecución** de las operaciones del sistema para cada combinación
de estructura (lista, ABB, B+), tamaño de datos N y tipo de datos (aleatorio u ordenado).

### 5.1 Hardware y software
Descritos en la sección 3. Los tiempos son válidos para ese entorno: en otro equipo cambiarán
los valores absolutos, aunque no necesariamente las tendencias.

### 5.2 Tamaños de entrada
Se usan N = 100, 1.000, 10.000, 100.000 y 1.000.000, cada uno 10 veces el anterior. Esta escala
cubre cuatro órdenes de magnitud con pocos puntos y deja los puntos a igual distancia en una
gráfica logarítmica, que es la forma habitual de estimar cómo crece un algoritmo con N
(Sedgewick y Wayne, 2011). Incluye N = 10.000, el caso del enunciado.
[Completa tras una prueba pequeña: si alguna combinación resulta demasiado lenta (por ejemplo,
el ABB con IDs ordenados), se reduce su N máximo y se declara aquí.]

Cada medición de búsqueda realiza **M = 1.000** búsquedas y se registra el tiempo total.
Una sola búsqueda es demasiado breve para medirla con fiabilidad; mil hacen el tiempo
medible incluso con N pequeño.

### 5.3 Generación de los datos
Los estudiantes se generan con `generar_estudiantes` en dos variantes:

| Tipo | IDs | Orden de inserción | Por qué se usa |
|---|---|---|---|
| Aleatorio | N IDs únicos tomados al azar entre 1 y 10·N (`random.sample`) | El orden del sorteo (aleatorio) | Caso típico: el ABB queda aproximadamente balanceado |
| Ordenado | 1, 2, …, N | Ascendente | Peor caso del ABB: se degenera en una cadena |

El rango hasta 10·N deja "huecos" entre los IDs, lo que permite buscar IDs que no existen.
Los demás campos no influyen en el costo de buscar (solo se comparan los IDs): el nombre se
elige al azar entre nombres generados con la librería Faker, la edad es un entero aleatorio
entre 17 y 30 y el promedio es un decimal aleatorio entre 0 y 10. Los IDs, edades y promedios
son reproducibles gracias a la semilla; los nombres no, porque Faker se usó sin semilla.

### 5.4 Generación de las búsquedas
- **Búsqueda por ID** (`generar_ids_busqueda`): lista de M IDs, el 80 % tomado al azar entre
  los que existen (con repetición) y el 20 % de IDs que no existen, todo mezclado. Así se mide
  tanto la búsqueda exitosa como la fallida. Con IDs ordenados, los inexistentes siempre quedan
  por encima de N, porque no hay huecos.
- **Búsqueda por rango** (`generar_rangos_por_tamano`): M rangos [lb, ub], cada uno con
  exactamente k = 50 estudiantes. Se fija k, en lugar del ancho en IDs, para que datos
  aleatorios y ordenados devuelvan la misma cantidad de resultados. Con IDs dispersos, un
  mismo ancho devolvería menos estudiantes que con IDs consecutivos, y la comparación mezclaría
  dos efectos.
- **Misma entrada para todos:** en cada repetición las tres estructuras reciben exactamente los
  mismos estudiantes y las mismas búsquedas, de modo que la comparación es justa.

### 5.5 Medición del tiempo
- **Reloj:** `time.perf_counter_ns()`, el reloj de mayor resolución disponible en Python para
  medir intervalos cortos, que devuelve enteros en nanosegundos (documentación de Python, módulo `time`).
- **Qué se cronometra:** solo la operación estudiada (las M búsquedas, el listado o, en el
  experimento de inserción, la construcción). Generar los datos y armar la estructura queda fuera.
- **Recolector de basura:** se desactiva durante la medición y se restaura después
  (`gc.disable()`), porque una pausa del recolector en medio de la medición agregaría un tiempo
  que no depende del algoritmo (documentación de Python, módulo `gc`).
- **Calentamiento:** las primeras 3 ejecuciones se descartan, porque suelen ser más lentas (cachés
  vacías, código aún sin cargar) y no representan el funcionamiento normal.
- **Unidad:** todo se mide en nanosegundos y se convierte a ms o s en las gráficas, indicando la unidad.

### 5.6 Repeticiones y reproducibilidad
Cada configuración se repite **30 veces**. Repetir es necesario porque la misma operación no
tarda siempre lo mismo (el sistema operativo y la caché introducen variación) y, con IDs
aleatorios, cada repetición usa datos distintos. Con 30 mediciones, el promedio se aproxima
bien a una distribución normal, aunque los tiempos individuales no lo sean (teorema central del
límite; la regla práctica de n ≥ 30 aparece en textos como Montgomery y Runger). [Si alguna
combinación es demasiado lenta, se reduce el número de repeticiones y se declara aquí.]

La repetición *i* usa la semilla `1000 + i`, de modo que cada una genera datos distintos pero
reproducibles. Cada medición se guarda como una fila de un archivo CSV (estructura, N, tipo
de datos, repetición, semilla, tiempo en ns), para poder recalcular las estadísticas sin volver
a medir.

### 5.7 Estadísticas utilizadas
Para cada configuración (30 tiempos) se calculan:

| Estadístico | Qué es | Para qué se usa |
|---|---|---|
| Media | Promedio de los tiempos | Valor típico; es lo que se grafica |
| Desviación estándar (muestral) | Cuánto se alejan los tiempos de la media | Mide el ruido o la variabilidad |
| Mediana | El valor central al ordenar los tiempos | Valor típico poco afectado por picos aislados |
| IQR | Q3 − Q1: ancho del 50 % central de los datos | Dispersión robusta; base para detectar atípicos |
| Intervalo de confianza del 95 % | media ± t · s / √n | Rango plausible para la media verdadera |

En el intervalo, *s* es la desviación estándar, *n* el número de repeticiones y *t* el valor de
la distribución t de Student con n − 1 grados de libertad (t ≈ 2,05 para n = 30). Se usa t y no
1,96 porque con pocas repeticiones la t es más exacta. **Cómo se lee:** si los intervalos de
dos estructuras no se solapan, la diferencia entre ellas probablemente es real; si se solapan,
no se puede afirmar que haya diferencia. La mediana se reporta además de la media porque los
tiempos tienden a tener picos hacia arriba (por ejemplo, cuando el sistema operativo ocupa la
CPU) que inflan la media.

### 5.8 Valores atípicos
Se usa el criterio de Tukey (1977): un tiempo es atípico si supera **Q3 + 1,5 · IQR** o es
menor que **Q1 − 1,5 · IQR**, calculado por separado en cada configuración. No se descarta nada
en silencio: de cada configuración se reporta cuántos atípicos hubo, y las estadísticas se
calculan **con y sin ellos**. Si las conclusiones son las mismas en ambos casos, no dependen de
esa decisión. Como cada repetición usa datos distintos, un valor alto no siempre es ruido (puede
ser un árbol que quedó más profundo), y por eso se muestran ambos resultados.

### 5.9 Comparación con la complejidad teórica
Si el tiempo crece como T(N) ≈ a · N^b, al graficar ambos ejes en escala logarítmica queda una
recta cuya **pendiente es el exponente b** (Sedgewick y Wayne, 2011). La pendiente se estima
con una regresión lineal sobre los logaritmos de N y de la media del tiempo:

| Pendiente observada | Crecimiento | Estructuras donde se espera |
|---|---|---|
| ≈ 0 (pequeña) | Constante o logarítmico | Búsqueda en ABB aleatorio y en B+ |
| ≈ 1 | Lineal (N log N da un poco más de 1) | Búsqueda en la lista; ABB ordenado |
| ≈ 2 | Cuadrático | Construcción del ABB con IDs ordenados |

Con costos fijos pequeños y efectos de caché, los tamaños más pequeños se desvían de la recta, por lo
que el ajuste se interpreta junto con la gráfica y no como un número aislado.